# A base model and a program specialist, live in one Serve app

## The story

A drug discovery team serves one **base model** to all its chemists. It is a general model: good on
many targets, not great on every one.

1. **A new program starts.** The team soaks a few hundred small "fragment" molecules into crystals of
   the target protein. X-rays show which fragments stick, and exactly how: tens of real answers
   (79 for EV-A71 2A).
2. **They test the base model** on those crystals. On this pocket it is often wrong.
3. **They fine-tune a copy** on the fragment crystals: the **specialist**. A small run, hours on a few
   GPUs. No new base model needed.
4. **They serve both.** Requests for this program go to the specialist. Everyone else keeps the base
   model.
5. **Not good enough yet?** They train a bigger specialist, give it a new version name and swap it in.
   No downtime.
6. **When the program ends,** its crystals go into the next big training run of the base model.

Why step 3 works: chemists grow the later compounds from the fragments, in the same pocket. What the
model learns from the fragments carries over. OpenBind measured it: 12% of later poses correct before
the fine-tune, 63% after.

## This demo

One compound: EV-A71 2A **x7259a**, a potent late follow-on (pKD 7.9). No model trained on it.
Boltz-2 adds the binding affinity at every step.

| Role | Deployment | Version | Checkpoint | x7259a, published RMSD |
|---|---|---|---|---|
| Base model | `BaseModel` | `p2-145k` | `of3-p2-145k.pt`, general OpenFold3 | 5.9 A, wrong |
| Specialist v1 | `EV71Specialist` | `ev71-v1` | Our Ray Train fine-tune of `of3-p2-145k.pt`: 8 L4s, 1,000 steps | 1.0 A, correct (measured live) |
| Specialist v2 | `EV71Specialist` | `ev71-v2` | `of3p2-ft-ev2a.ckpt`, OpenBind fine-tune: 8 GPUs, ~1,000 steps | 0.5 A, correct |

**Why not OpenBind-0 as the base model?** It already gets x7259a right (0.40 A) and 83% of EV-A71 2A
poses. It may have trained on this data. A base model that already knows the target leaves nothing to
fix.

## How it works in Ray Serve

- One class, `OF3Live`, is deployed twice: `BaseModel` and `EV71Specialist`. Each has its own
  replicas and its own checkpoint.
- The `Viewer` sends each request to one of them. `pose_from` in its `user_config` sets the default;
  a request can name another one.
- A swap is an in-place update that changes only `user_config`. Ray Serve calls `reconfigure()` on the
  running replicas. No restart.
- Each version keeps its poses in its own cache folder. Never reuse a version name for new weights,
  and never use `base`.

## Real-world scale (rough estimates)

| | Small biotech | Mid-size biotech | Big pharma |
|---|---|---|---|
| Chemists using it | 2 to 10 | 20 to 100 | 500 or more |
| Programs at once | 1 to 3 | 5 to 15 | 50 or more |
| Live pose requests per day | tens to ~100 | hundreds to a few thousand | thousands to tens of thousands |

- A chemist checks maybe 10 to 50 ideas a day. Making a compound takes days, so live traffic is small.
- Tools that propose thousands of molecules a day are batch work: Ray Data, not this endpoint.
- Fragment screens are small: hundreds soaked, tens to ~100 seen bound. 100,000+ molecules is a
  virtual screen, done on a computer.
- One OpenFold3 prediction is ~10 GPU-seconds in batch, 1 to 2 minutes live here. The hard part is
  many models with uneven traffic, not the total volume.

## How to scale it

- Each deployment scales on its own queue (`autoscaling_config`).
- **Base model:** most of the traffic, so the most replicas.
- **A specialist:** one program's chemists, so few replicas. `min_replicas: 0` frees the GPU when idle,
  but the first request after that is slow.
- **Many specialists:** Ray Serve model multiplexing serves many checkpoints from one deployment, and
  routes each request to a replica that already has its model loaded.
- **GPUs from any zone:** `enable_cross_zone_scaling` in `jobs/service.yaml`.
- **This workshop Service** is fixed at 2 L4s: four half-GPU slots, up to two for Boltz-2 and one for
  each OpenFold3 deployment.

## Before you start

- Run this notebook in a Workspace, from the repository folder. It deploys `jobs/service.yaml` from here.
- The data and checkpoints must be in `/mnt/shared_storage/aiddd`. If `of3-p2-145k.pt` is missing, run
  `anyscale job submit -f job.yaml -- python scripts/setup_data.py`.
- A new molecule takes one to three minutes per step. Run all steps once before the session, so each
  step returns from its cache in seconds.
- Do not change files in this folder between the first deploy and the swaps. A code change makes the
  update restart every replica. Notebooks are excluded from the upload, so editing this one is safe.

In [ ]:
import json, os, time
from pathlib import Path

import anyscale
import pandas as pd
import requests
from anyscale.service.models import ServiceConfig

SHARED = Path(os.environ.get("AIDDD_SHARED", "/mnt/shared_storage/aiddd"))
CKPTS = SHARED / "of3_cache"
assert (SHARED / "results" / "ev71-2a").exists(), f"missing {SHARED}/results: copy the data first"
assert (CKPTS / "of3-p2-145k.pt").exists(), "missing of3-p2-145k.pt: run scripts/setup_data.py"

# Your own Service. "aiddd-viewer" is the shared workshop Service: never use it for practice.
SERVICE_NAME = "aiddd-viewer-geoff"
assert SERVICE_NAME != "aiddd-viewer", "pick your own Service name for practice runs"

# x7259a: the base model is wrong, the public fine-tune is right. Backups: x7570a, x7056a.
SMILES = "CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1"

# Specialist v1: our Ray Train fine-tune of of3-p2-145k.pt (66 fragments, 1,000 steps, 8 L4s). None skips it.
OURS = SHARED / "finetune/runs/ft-p2-evfrag-1000s-8g-1023-0139/checkpoint_2026-10-23_04-05-55.886640/of3_ev71_ft.pt"
assert OURS is None or Path(OURS).exists(), f"missing {OURS}"


def of3(version, name, checkpoint):
    """user_config of an OpenFold3 deployment. All demo checkpoints are OF3-p2: legacy OpenFold3."""
    return {"version": version, "name": name, "checkpoint": str(checkpoint), "legacy": True}


BASE = of3("p2-145k", "Base model: OF3-p2 145k", CKPTS / "of3-p2-145k.pt")
SPECIALISTS = {"ev71-v2": of3("ev71-v2", "EV-A71 specialist v2: public fine-tune (8 GPUs, 1,000 steps)",
                              CKPTS / "of3p2-ft-ev2a.ckpt")}
if OURS:
    SPECIALISTS = {"ev71-v1": of3("ev71-v1", "EV-A71 specialist v1: our Ray Train fine-tune (8 L4s, 1,000 steps)", OURS),
                   **SPECIALISTS}

# One demo step: the user_config each deployment it touches gets. Others keep their defaults.
STEPS = {"base": {"Viewer": {"pose_from": "BaseModel"}, "BaseModel": BASE}}
for version, cfg in SPECIALISTS.items():
    STEPS[version] = {"Viewer": {"pose_from": "EV71Specialist"}, "BaseModel": BASE, "EV71Specialist": cfg}
list(STEPS)

## Helpers

`service_config()` is `jobs/service.yaml` with your Service name and a `user_config` for each listed
deployment. `predict()` calls the Viewer's `/predict` endpoint, the one the web page uses. `swap_to()`
makes one in-place update, then sends the molecule until the answer comes from the new model.

In [ ]:
def service_config(user_configs):
    """service.yaml, renamed, with {deployment name: user_config} overrides."""
    cfg = ServiceConfig.from_yaml("jobs/service.yaml", name=SERVICE_NAME)
    deployments = [{"name": d, "user_config": uc} for d, uc in user_configs.items()]
    return cfg.options(applications=[{**app, "deployments": deployments} for app in cfg.applications])


def predict(smiles, pose_from=None):
    """pose_from: None uses the Viewer's default route, or name a deployment."""
    s = anyscale.service.status(name=SERVICE_NAME)
    headers = {"Authorization": f"Bearer {s.query_auth_token}"} if s.query_auth_token else {}
    body = {"smiles": smiles, **({"pose_from": pose_from} if pose_from else {})}
    r = requests.post(s.query_url.rstrip("/") + "/predict", json=body, headers=headers, timeout=600)
    return r.json()


def summary(out):
    """The model behind each part of one /predict response, and the main numbers."""
    models, key, m = out.get("models") or {}, out.get("answer_key") or {}, out.get("metrics", {})
    return {"routed to": out.get("pose_from"),
            "pose model": f"{models.get('pose', {}).get('name')} ({models.get('pose', {}).get('version')})",
            "affinity model": f"{models.get('affinity', {}).get('name')} ({models.get('affinity', {}).get('version')})",
            "crystal": key.get("complex"), "ligand RMSD (A)": key.get("lig_rmsd"),
            "pose ipTM": out.get("pose_iptm"), "affinity (log10 IC50, uM)": m.get("affinity_pred_value"),
            "status": out.get("status"), "seconds": out.get("seconds")}


responses = {}


def swap_to(step):
    """In-place update to one step, then the molecule until the answer comes from its model."""
    if step not in STEPS:
        return {"skipped": f"{step}: no checkpoint yet"}
    route = STEPS[step]["Viewer"]["pose_from"]
    version = STEPS[step][route]["version"]
    t0 = time.time()
    anyscale.service.deploy(service_config(STEPS[step]), in_place=True, name=SERVICE_NAME)
    anyscale.service.wait(name=SERVICE_NAME, state="RUNNING", timeout_s=600)
    for _ in range(30):
        out = predict(SMILES)
        if out.get("pose_from") == route and (out.get("models") or {}).get("pose", {}).get("version") == version:
            break
        time.sleep(5)
    else:
        raise TimeoutError(f"the answer still does not come from {route} {version}")
    print(f"swap and prediction took {time.time() - t0:.0f} s")
    responses[step] = out
    return summary(out)

## 1. Start the Service: base model only

The first deploy starts a new cluster with two L4 nodes, about 5 to 10 minutes. If the Service is
already running, this cell only resets it, in place.

In [ ]:
try:
    running = str(anyscale.service.status(name=SERVICE_NAME).state) == "RUNNING"
except Exception:   # no Service with this name yet
    running = False
anyscale.service.deploy(service_config(STEPS["base"]), in_place=running, name=SERVICE_NAME)
anyscale.service.wait(name=SERVICE_NAME, state="RUNNING", timeout_s=1800)
print(anyscale.service.status(name=SERVICE_NAME).query_url)

## 2. The base model on the new program's compound

Every request goes to the base model. It has never seen this pocket with a ligand in it.

In [ ]:
swap_to("base")

## 3. Add the specialist, v1

Our fine-tune of the base model on the EV-A71 2A fragment crystals, with `finetune_ray_train.py`.
EV-A71 2A requests now go to it.

In [ ]:
swap_to("ev71-v1")

## 4. Not good enough: swap in specialist v2

A bigger fine-tune of the same base model, with a new version name: OpenBind's public checkpoint,
8 GPUs, about 1,000 steps, 32 diffusion samples per step.

In [ ]:
swap_to("ev71-v2")

## 5. Both models, one Service

The base model still serves every other request. The same molecule, sent to each one:

In [ ]:
both = {name: predict(SMILES, pose_from=name) for name in ("BaseModel", "EV71Specialist")}
pd.DataFrame({name: summary(out) for name, out in both.items()})

## 6. Every step side by side

In [ ]:
pd.DataFrame({step: summary(out) for step, out in responses.items()})

The full response metadata of the last step, as the web page receives it:

In [ ]:
last = list(responses.values())[-1]
print(json.dumps({k: last.get(k) for k in ("pose_from", "models", "answer_key", "pose_model", "pose_iptm")}, indent=2))

## 7. Back to the start

The same in-place update, back to the base model only. It waits until the update is done, so a rerun
from the top does not start while this update is still in progress.

In [ ]:
anyscale.service.deploy(service_config(STEPS["base"]), in_place=True, name=SERVICE_NAME)
anyscale.service.wait(name=SERVICE_NAME, state="RUNNING", timeout_s=600)

## 8. Stop the Service

The Service keeps two L4 nodes running until you stop it.

In [ ]:
# anyscale.service.terminate(name=SERVICE_NAME)